# Phase 6.2 — Master Analysis

*Real-only notebook: loads the latest persisted experiment report JSON and analyzes it. No simulations.*


## Load latest RQ reports (strict, no fallback)

In [ ]:
import json
from pathlib import Path

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 200)

def latest_report(report_dir: Path, pattern: str) -> Path:
    files = sorted(report_dir.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
    if not files:
        raise FileNotFoundError(f'Missing required report: {pattern} in {report_dir.resolve()}')
    return files[0]

paths = {
    'rq1': latest_report(Path('../results/rq1'), 'rq1_report_*.json'),
    'rq2': latest_report(Path('../results/rq2'), 'rq2_report_*.json'),
    'rq3': latest_report(Path('../results/rq3'), 'rq3_report_*.json'),
    'rq4': latest_report(Path('../results/rq4'), 'rq4_report_*.json'),
    'rq5': latest_report(Path('../results/rq5'), 'rq5_report_*.json'),
}

reports = {k: json.loads(p.read_text(encoding='utf-8')) for k, p in paths.items()}
print('Using:')
for k, p in paths.items():
    print(' -', k, p)


## Quick summary tables

In [ ]:
# RQ1
rq1_agg = pd.DataFrame.from_dict(reports['rq1'].get('aggregate_metrics') or {}, orient='index').reset_index(names='llm_model')
if not rq1_agg.empty:
    rq1_agg = rq1_agg.sort_values('f1_mean', ascending=False)

# RQ2
rq2 = reports['rq2']
rq2_summary = pd.DataFrame([{
    'total_endpoints': rq2.get('total_endpoints'),
    'avg_coherence_score': rq2.get('avg_coherence_score'),
    'total_inconsistencies': rq2.get('total_inconsistencies'),
    'inconsistencies_per_endpoint': rq2.get('inconsistencies_per_endpoint'),
}])

# RQ3
rq3_agg = pd.DataFrame.from_dict(reports['rq3'].get('aggregate_metrics') or {}, orient='index').reset_index(names='llm_model')
if not rq3_agg.empty:
    rq3_agg = rq3_agg.sort_values('overall_mean', ascending=False)

# RQ4
rq4_models = pd.DataFrame([
    {'llm_model': r.get('llm_model'), 'overall_score': r.get('overall_score'),
     'oracle_f1': (r.get('oracle_metrics') or {}).get('f1'),
     'coherence': (r.get('consistency_metrics') or {}).get('coherence_score'),
     'time_ms': (r.get('performance_metrics') or {}).get('avg_generation_time_ms'),
     'cost_usd': (r.get('cost_metrics') or {}).get('cost_per_endpoint_usd')}
    for r in (reports['rq4'].get('model_results') or [])
]).sort_values('overall_score', ascending=False)

# RQ5
rq5_findings = reports['rq5'].get('overall_findings') or {}
rq5_summary = pd.DataFrame([rq5_findings])

print('RQ1 aggregate:')
display(rq1_agg)
print('RQ2 summary:')
display(rq2_summary)
print('RQ3 aggregate:')
display(rq3_agg)
print('RQ4 model results:')
display(rq4_models)
print('RQ5 overall findings:')
display(rq5_summary)


## Cross-RQ visualization (best overall score in RQ4)

In [ ]:
if not rq4_models.empty:
    plt.figure(figsize=(10, 4))
    sns.barplot(data=rq4_models, x='llm_model', y='overall_score')
    plt.title('RQ4 — Overall score by LLM (master view)')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()


## Export

In [ ]:
OUT_DIR = Path('./outputs/master')
OUT_DIR.mkdir(parents=True, exist_ok=True)

rq1_agg.to_csv(OUT_DIR / 'rq1_aggregate_metrics.csv', index=False)
rq2_summary.to_csv(OUT_DIR / 'rq2_summary.csv', index=False)
rq3_agg.to_csv(OUT_DIR / 'rq3_aggregate_metrics.csv', index=False)
rq4_models.to_csv(OUT_DIR / 'rq4_model_results.csv', index=False)
rq5_summary.to_csv(OUT_DIR / 'rq5_overall_findings.csv', index=False)

meta = {
    'reports': {k: str(v) for k, v in paths.items()},
    'rq4_best_models': reports['rq4'].get('best_models'),
    'rq5_recommendations': reports['rq5'].get('recommendations'),
}
(OUT_DIR / 'master_summary.json').write_text(json.dumps(meta, indent=2), encoding='utf-8')

print('Wrote:', OUT_DIR)
